# jevmark v3: learners from a deployment-feedback log on Banking77 (tasks 3.4 and 3.5, decision 56)

Thin wrapper: clone the private repo at one commit, install pinned dependencies, build the nine v1.3 splits and the v3 Banking77 files, copy the sft_06b adapter from the attached adapter dataset, then run one session of the v3 plan (`SESSION`, docs/KAGGLE.md section 12). Session A evaluates sft_06b on the v3 splits (the zero-shot learner, `runs/v3_06b_zeroshot`), collects the two deployment logs (seeds 0 and 1) and trains full_sft and positive_sft at N 500, 2000 and 5000. Sessions B to E copy the logs from the attached log dataset, check each log's sha256 against `LOG_SHA256`, and train their runs: RLCD and noisy runs in B and C, full_sft at N 5000 for training seeds 1 and 2 in D, and in E the soft-target noise diagnostic (direct_brier at N 5000 with `--noise soft`, decision 59), whose final `adapter_last` is evaluated too (task 3.6). Every run is trained, checked (TRAINING PASS or FAIL), evaluated on v3_banking77_test_full, test_banking77, test_indomain and test_unseen_intents, and copied to `/kaggle/working/runs` as it finishes, with one RUN DONE line. A failure stops the session; runs already finished stay in the output. All logic lives in the repo (`scripts/collect_log.py`, `scripts/train_rlcd.py` log mode, `scripts/evaluate.py`).

Settings: accelerator GPU T4 x2 (one is used), internet on, secret `GITHUB_TOKEN` attached, the adapter dataset added as input (`ADAPTER_DATASET`), and for sessions B to E the log dataset (`LOG_DATASET`).

In [ ]:
# Parameters: set before running. COMMIT must be a full 40-character sha, the same for sessions A to D (session D's plan lives in this notebook, so it runs at the commit of A to C); session E needs --noise soft, so it runs at the task 3.6 soft-target commit or later (docs/KAGGLE.md section 12).
REPO = "OWNER/jevmark"
COMMIT = "0000000000000000000000000000000000000000"
SESSION = "A"         # "A" to "E" (docs/KAGGLE.md section 12)
ADAPTER_DATASET = "/kaggle/input/jevmark-sft-adapters"  # holds runs/sft_06b/adapter (docs/KAGGLE.md section 10)
LOG_DATASET = "/kaggle/input/jevmark-v3-logs"  # sessions B to E: holds v3_log_s0/ and v3_log_s1/ from session A
LOG_SHA256 = {0: "", 1: ""}  # sessions B to E: each log's sha256 as session A printed it (LOG DONE lines)
SMOKE_STEPS = 10      # optimizer steps in the smoke run
MAX_HOURS = 2.0       # training wall clock cap per run (the longest, 1407 steps, takes about 1.4 h)


In [ ]:
# Clone REPO at COMMIT. The token reaches git only through environment variables,
# is never put on a command line or in .git/config, and is redacted from any output.
import base64
import os
import re
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

assert re.fullmatch(r"[\w.-]+/[\w.-]+", REPO), "REPO must be owner/name"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "COMMIT must be a full 40-character sha"
assert SESSION in ("A", "B", "C", "D", "E"), "SESSION must be A, B, C, D or E"
# Let the CUDA caching allocator grow segments instead of fragmenting (decision 45). PyTorch 2.9 and later
# read PYTORCH_ALLOC_CONF; earlier releases read only PYTORCH_CUDA_ALLOC_CONF, so both are set. Every later
# !python inherits them.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK = Path("/tmp/jevmark")  # outside /kaggle/working, so the notebook output holds only runs/


def run(cmd, env=None, secrets=()):
    result = subprocess.run(cmd, cwd=WORK, env=env, capture_output=True, text=True)
    output = result.stdout + result.stderr
    for secret in secrets:
        output = output.replace(secret, "***")
    if output.strip():
        print(output[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"{cmd[0]} {cmd[1]} failed with exit code {result.returncode}")


token = UserSecretsClient().get_secret("GITHUB_TOKEN")
header = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = {
    **os.environ,
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": header,
}
WORK.mkdir(parents=True, exist_ok=True)
if not (WORK / ".git").exists():
    run(["git", "init", "-q"])
    run(["git", "remote", "add", "origin", f"https://github.com/{REPO}.git"])
run(["git", "fetch", "-q", "--depth", "1", "origin", COMMIT], env=git_env, secrets=(token, header))
run(["git", "checkout", "-q", "--force", "FETCH_HEAD"])
del token, header, git_env

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=WORK, capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT, f"checked out {head}, expected {COMMIT}"
os.chdir(WORK)
print("checked out", head)

# The session plan (docs/KAGGLE.md section 12): (arm, N, training seed, noisy, log seed) per run, in order.
PLAN = {
    "A": [("full_sft", 500, 0, False, 0), ("full_sft", 2000, 0, False, 0), ("full_sft", 5000, 0, False, 0),
          ("positive_sft", 500, 0, False, 0), ("positive_sft", 2000, 0, False, 0), ("positive_sft", 5000, 0, False, 0)],
    "B": [("direct_brier", 500, 0, False, 0), ("direct_brier", 2000, 0, False, 0), ("direct_brier", 5000, 0, False, 0), ("direct_brier", 5000, 1, False, 0)],
    "C": [("direct_brier", 5000, 2, False, 0), ("direct_brier", 5000, 0, True, 0), ("positive_sft", 5000, 0, True, 0), ("direct_brier", 5000, 0, False, 1)],
    "D": [("full_sft", 5000, 1, False, 0), ("full_sft", 5000, 2, False, 0)],  # task 3.6: full_sft training seeds for prediction 3
    "E": [("direct_brier", 5000, 0, "soft", 0)],  # task 3.6: the soft-target noise diagnostic (decision 59)
}[SESSION]
# The noisy field: False (clean), True (the stored flipped outcome; --noisy, which every commit accepts) or "soft".
NOISE_FLAG = {False: "", True: "--noisy", "soft": "--noise soft"}
NOISE_SUFFIX = {False: "", True: "_noisy", "soft": "_noisy_soft"}
EVAL_LAST = SESSION == "E"  # also evaluate each run's final adapter_last as runs/<run>_last


def run_name_of(arm, n, seed, noisy, log):
    """The run name train_rlcd.py gives in log mode."""
    return f"v3_06b_{arm}_n{n}_s{seed}" + NOISE_SUFFIX[noisy] + (f"_log{log}" if log else "")


RUNS = [run_name_of(*plan) for plan in PLAN]
SMOKE_RUN = f"{RUNS[0]}_smoke"
LOGS = sorted({plan[4] for plan in PLAN} | ({0, 1} if SESSION == "A" else set()))
if SESSION != "A":
    assert all(re.fullmatch(r"[0-9a-f]{64}", LOG_SHA256.get(k, "")) for k in LOGS), f"set LOG_SHA256 for logs {LOGS} from session A's LOG DONE lines"

# Committed run directories are history, not state (decision 45): delete every directory this session writes.
import shutil

stale = [*RUNS, SMOKE_RUN, *(["v3_06b_zeroshot", "v3_log_s0", "v3_log_s1"] if SESSION == "A" else []), *([f"{r}_last" for r in RUNS] if EVAL_LAST else [])]
for name in stale:
    shutil.rmtree(WORK / "runs" / name, ignore_errors=True)
print("session", SESSION, "runs", ", ".join(RUNS))
print("removed committed run directories:", ", ".join(stale))


In [ ]:
# The Hugging Face packages pinned to uv.lock; Kaggle keeps its own torch and numpy (decision 23). jevmark itself without deps.
# Kaggle's preinstalled torchao 0.10 makes peft 0.21 raise on LoRA adapter injection; jevmark does not use it.
!pip uninstall -y -q torchao
!pip install -q -r requirements-kaggle.txt
!pip install -q -e . --no-deps
!python -c "import sys, torch, transformers, peft, datasets; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'transformers', transformers.__version__, 'peft', peft.__version__, 'datasets', datasets.__version__)"

In [ ]:
# Fail-fast (decision 45): a failing !python does not stop a cell, so every training cell checks its own result
# with jevmark.runcheck and raises on failure, and every evaluation cell refuses to run unless its training passed.
import datetime
import json
import shutil

from jevmark.runcheck import require_training

TRAINED = {}  # "smoke" and each run name -> True once its training printed TRAINING PASS


def evaluation_allowed(stage):
    if not TRAINED.get(stage):
        raise RuntimeError(f"refusing to evaluate: the {stage} training cell did not print TRAINING PASS")


def check_exit(what, code):
    if code != 0:
        raise RuntimeError(f"{what} exited with code {code}")


In [ ]:
# Build the nine v1.3 splits, then the v3 Banking77 files; build_v3_data.py checks the v1.3 files against
# data/baseline_subset.json and the v3 files against the sha256 recorded in configs/v3_data.yaml, and fails on any difference.
!make data-build PY=python
!python scripts/build_v3_data.py
check_exit("build_v3_data.py", _exit_code)


In [ ]:
# Copy the sft_06b adapter from the adapter dataset into the clone: runs/sft_06b/adapter. It is the logging policy,
# the zero-shot learner and the starting point of every learner; each run records its sha256.
import hashlib

candidates = [Path(ADAPTER_DATASET) / "runs" / "sft_06b" / "adapter", Path(ADAPTER_DATASET) / "sft_06b" / "adapter"]
source = next((c for c in candidates if (c / "adapter_model.safetensors").is_file()), None)
if source is None:
    raise RuntimeError(f"no sft_06b adapter under {ADAPTER_DATASET}; looked in {[str(c) for c in candidates]}")
target = WORK / "runs" / "sft_06b" / "adapter"
shutil.rmtree(target, ignore_errors=True)
shutil.copytree(source, target)
print("sft_06b adapter", source, "->", target, "sha256", hashlib.sha256((target / "adapter_model.safetensors").read_bytes()).hexdigest())


In [ ]:
# The deployment logs. Session A: evaluate sft_06b on the v3 splits (runs/v3_06b_zeroshot, the zero-shot learner),
# then collect the logs of seeds 0 and 1 (runs/v3_log_s<k>/, about 10 minutes each), copying each out as it finishes.
# Sessions B to E: copy the logs from LOG_DATASET and check each log's sha256 against its metrics.json and LOG_SHA256.
OUT = Path("/kaggle/working/runs")
OUT.mkdir(parents=True, exist_ok=True)


def copy_run(run_name):
    shutil.copytree(WORK / "runs" / run_name, OUT / run_name, dirs_exist_ok=True)


def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


if SESSION == "A":
    !python scripts/evaluate.py --ckpt runs/sft_06b --run-name v3_06b_zeroshot --device cuda
    check_exit("evaluate.py for v3_06b_zeroshot", _exit_code)
    copy_run("v3_06b_zeroshot")
    zero = json.loads((OUT / "v3_06b_zeroshot" / "metrics.json").read_text())
    assert zero["git"]["commit"] == COMMIT and set(zero["splits"]) == {"v3_banking77_test_full", "test_banking77", "test_indomain", "test_unseen_intents"}
    o = zero["splits"]["v3_banking77_test_full"]["overall"]
    print(f"ZERO-SHOT DONE: v3_banking77_test_full acc {o['accuracy']:.4f} ece {o['ece']:.4f}", flush=True)
    for k in LOGS:
        !python scripts/collect_log.py --ckpt runs/sft_06b --seed {k} --device cuda
        check_exit(f"collect_log.py seed {k}", _exit_code)
        copy_run(f"v3_log_s{k}")
        m = json.loads((OUT / f"v3_log_s{k}" / "metrics.json").read_text())
        assert m["git"]["commit"] == COMMIT and m["log_sha256"] == file_sha256(OUT / f"v3_log_s{k}" / "log.jsonl")
        LOG_SHA256[k] = m["log_sha256"]
        print(f"LOG DONE seed {k}: sha256 {m['log_sha256']}, {m['n']} interactions, accuracy {m['accuracy']:.4f}, outcome rate {m['outcome_rate']:.4f}, "
              f"flipped {m['flipped_outcome_rate']:.4f}, chosen other {m['chosen_other_rate']:.4f}", flush=True)
else:
    for k in LOGS:
        candidates = [Path(LOG_DATASET) / "runs" / f"v3_log_s{k}", Path(LOG_DATASET) / f"v3_log_s{k}"]
        source = next((c for c in candidates if (c / "log.jsonl").is_file() and (c / "metrics.json").is_file()), None)
        if source is None:
            raise RuntimeError(f"no v3_log_s{k}/log.jsonl and metrics.json under {LOG_DATASET}; looked in {[str(c) for c in candidates]}")
        target = WORK / "runs" / f"v3_log_s{k}"
        shutil.rmtree(target, ignore_errors=True)
        shutil.copytree(source, target)
        digest = file_sha256(target / "log.jsonl")
        recorded = json.loads((target / "metrics.json").read_text())["log_sha256"]
        if digest != recorded or digest != LOG_SHA256[k]:
            raise RuntimeError(f"log seed {k}: sha256 {digest}, its metrics.json records {recorded}, LOG_SHA256 says {LOG_SHA256[k]}; not the log session A wrote")
        print(f"LOG OK seed {k}: {source} -> {target}, sha256 {digest}", flush=True)


In [ ]:
# Smoke run on the first run of the plan: SMOKE_STEPS steps into runs/{SMOKE_RUN}/ (gitignored), with the pre-flight memory
# check and the step-0 selection. Prints TRAINING PASS or TRAINING FAIL and raises on FAIL.
ARM, N, SEED, NOISY, LOG = PLAN[0]
NOISY_FLAG = NOISE_FLAG[NOISY]
started = datetime.datetime.now(datetime.timezone.utc)
!python scripts/train_rlcd.py --config configs/v3_06b.yaml --log runs/v3_log_s{LOG}/log.jsonl --n {N} --init runs/sft_06b arm={ARM} seed={SEED} run_name={SMOKE_RUN} {NOISY_FLAG} --limit-steps {SMOKE_STEPS} --device cuda
require_training(WORK / "runs" / SMOKE_RUN, started, _exit_code)
TRAINED["smoke"] = True


In [ ]:
# Every run of the plan, in order: training from the log into runs/<run>/, the evaluation of its best adapter on
# the v3 splits (v3_banking77_test_full, test_banking77, test_indomain, test_unseen_intents: the default for a v3 run),
# then an immediate copy of that run (adapter/, adapter_last/, last/ included) to /kaggle/working/runs and one RUN DONE
# line. TRAINING FAIL or an evaluation error raises and stops the session; the runs finished before it are already in
# the output. A run stopped at MAX_HOURS leaves runs/<run>/last in the output for --resume (docs/KAGGLE.md section 10).
if not TRAINED.get("smoke"):
    raise RuntimeError("refusing to train: the smoke run did not print TRAINING PASS")
DONE = []  # one RUN DONE line per finished run


def run_summary(run_name, log, summary, run_started):
    metrics = json.loads((OUT / run_name / "metrics.json").read_text())
    assert metrics["git"]["commit"] == COMMIT, run_name
    assert metrics["v3"]["log_sha256"] == summary["log_sha256"] == LOG_SHA256[log], f"{run_name}: not trained on the checked log"
    assert metrics["init_adapter_sha256"] == summary["init_adapter_sha256"], f"{run_name}: evaluated run and training summary name different adapters"
    full, indomain = metrics["splits"]["v3_banking77_test_full"]["overall"], metrics["splits"]["test_indomain"]["overall"]
    minutes = (datetime.datetime.now(datetime.timezone.utc) - run_started).total_seconds() / 60
    return (f"RUN DONE {run_name}: best step {summary['best_step']} of {summary['steps']}, {summary['n_used']} interactions used, "
            f"v3_banking77_test_full acc {full['accuracy']:.4f} ece {full['ece']:.4f}, test_indomain acc {indomain['accuracy']:.4f}, wall clock {minutes:.1f} min")


for (arm, n, seed, noisy, log), run_name in zip(PLAN, RUNS):
    noisy_flag = NOISE_FLAG[noisy]
    run_started = datetime.datetime.now(datetime.timezone.utc)
    print(f"=== {run_name}: training", flush=True)
    !python scripts/train_rlcd.py --config configs/v3_06b.yaml --log runs/v3_log_s{log}/log.jsonl --n {n} --init runs/sft_06b arm={arm} seed={seed} {noisy_flag} --max-hours {MAX_HOURS} --device cuda
    training_exit = _exit_code
    copy_run(run_name)  # keep the state even if the check fails
    summary = require_training(WORK / "runs" / run_name, run_started, training_exit)
    TRAINED[run_name] = True
    evaluation_allowed(run_name)
    print(f"=== {run_name}: evaluating", flush=True)
    !python scripts/evaluate.py --ckpt runs/{run_name} --device cuda
    check_exit(f"evaluate.py for {run_name}", _exit_code)
    copy_run(run_name)
    DONE.append(run_summary(run_name, log, summary, run_started))
    print(DONE[-1], flush=True)
    if EVAL_LAST:
        # The final adapter as its own run (scripts/prepare_adapter_run.py records its source and sha256), evaluated on
        # the same splits, so the selected and the final step can be compared (decision 59).
        last_name = f"{run_name}_last"
        print(f"=== {last_name}: evaluating adapter_last", flush=True)
        !python scripts/prepare_adapter_run.py --adapter runs/{run_name}/adapter_last --source-run runs/{run_name} --adapter-dir adapter_last --name {last_name}
        check_exit(f"prepare_adapter_run.py for {last_name}", _exit_code)
        !python scripts/evaluate.py --ckpt runs/{last_name} --device cuda
        check_exit(f"evaluate.py for {last_name}", _exit_code)
        copy_run(last_name)
        last = json.loads((OUT / last_name / "metrics.json").read_text())
        assert last["git"]["commit"] == COMMIT and last["adapter_sha256"] == file_sha256(OUT / run_name / "adapter_last" / "adapter_model.safetensors"), last_name
        full = last["splits"]["v3_banking77_test_full"]["overall"]
        DONE.append(f"LAST DONE {last_name}: adapter_last of {run_name}, adapter sha256 {last['adapter_sha256']}, v3_banking77_test_full acc {full['accuracy']:.4f} ece {full['ece']:.4f}")
        print(DONE[-1], flush=True)


In [ ]:
# Recap of the finished runs: the RUN DONE lines, then each run's provenance and its four splits.
for line in DONE:
    print(line)
for run_name in RUNS[: len(DONE)]:
    metrics = json.loads((OUT / run_name / "metrics.json").read_text())
    summary = json.loads((OUT / run_name / "train_summary.json").read_text())
    print(run_name, "commit", metrics["git"]["commit"][:12], "dirty", metrics["git"]["dirty"], "fp32 fallback", metrics["precision"]["fp32_fallback_used"],
          "log", summary["log_sha256"][:12], "best step", summary["best_step"], f"{summary['wall_clock_seconds_this_session'] / 60:.1f} min training")
    for split in ("v3_banking77_test_full", "test_banking77", "test_indomain", "test_unseen_intents"):
        o = metrics["splits"][split]["overall"]
        print(f"  {split:24} acc {o['accuracy']:.4f} ece {o['ece']:.4f} nll {o['nll']:.4f}")
